**Playground copy of `Projects/GKP_cavity/notebooks/Sisyphus_A.ipynb`** (made by Claude, 2026-11-13).

Used only to test the pipeline with `tools/nbrun.py`. Not a design result.

* Data paths point to `Claude/Design_notebooks/Sisyphus_style/playground/data/` (copies of the Sisyphus_A dictionaries; the copies now contain coarse test values).
* Cells with outputs are the ones that were run: setup and drawing, Q3D capacitance, bare filter eigenmode, qubit BBQ to storage, pin S-parameter of the filter, the Save cells, Summary/Chi.
* Lines marked `TEST` / `TEST MODE (Claude)` are coarse settings (100 um meshes, few passes) and the restore of the template setups.
* `chip_y` is 18500 here instead of 18800: with 18800 the chip box overlaps the template solid `wafer_edge` and HFSS refuses to solve.
* The cleanup cells are still the original ones (without the `if rendered:` guard described in `sisyphus_style.md`). Do not run a cleanup cell here when nothing is rendered.

In [1]:
%load_ext autoreload
%autoreload 2
%load_ext autotime

import sys
import logging
import pandas as pd
from IPython.display import display
import pickle
import matplotlib.pyplot as plt
import numpy as np
import scipy as scp
import tqdm
from ansys.aedt.core import Hfss
from ansys.aedt.core import Q3d
from ansys.aedt.core import generic
ansys_version=252
import pickle
import numdifftools as nd
from resonator_tools import circuit as rfit

# folder path
# system = "Z:/_USERS/AndreiK/ 
# PLAYGROUND COPY: local paths, the originals on B: are never read or written
system = "C:/Users/akugut/Documents/AndreiK_backup/"

data_path = system+"Claude/Design_notebooks/Sisyphus_style/playground/data/"
gds_path = system+"Claude/Design_notebooks/Sisyphus_style/playground/data/"

import tiny_useful_lib as tul
import qiskit_metal as metal

from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs

from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.framed_path import RouteFramed
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround

from extra_elements.jjs import Patch
from extra_elements.jjs import JunctionManhattan_cross
from extra_elements.jjs import Taper
from extra_elements.cpw import CirclePad
from extra_elements.general import TransmonPocketTeethCustom

# turn off INFO output
logging.disable(logging.INFO)


time: 3.64 s (started: 2026-11-13 17:49:28 +01:00)


In [2]:
def T_ph(T1, T2): return 1/(1/T2 - 1/2/T1)
T_ph(47, 10.8)

12.201923076923078

time: 16 ms (started: 2026-11-13 17:49:31 +01:00)


In [3]:
# setup experimental variables
exp_vars = {}
exp_vars.update({'fudge_factor' : 1/(1 - 0.004)})
exp_vars.update({'cpw_c_fF/mm' : 169.4})
exp_vars.update({'Ej_to_S' : 400})
exp_vars.update({'evaporation_shift' : 1})
exp_vars.update({'undercut' : 0.2})

# other parameters dicts
Ej_dict = {}
Ej_dict.update({'T' : 24})
Ec_dict = {}
g_dict = {}
Qe_dict = {}
Qi_dict = {}
Q_dict = {}
f_dict = {}

# load
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'rb') as f: f_dict = pickle.load(f)
with open(data_path + 'Sisyphus_A_Q_dict.pkl', 'rb') as f: Q_dict = pickle.load(f)
with open(data_path + 'Sisyphus_A_Qi_dict.pkl', 'rb') as f: Qi_dict = pickle.load(f)
with open(data_path + 'Sisyphus_A_Qe_dict.pkl', 'rb') as f: Qe_dict = pickle.load(f)
with open(data_path + 'Sisyphus_A_g_dict.pkl', 'rb') as f: g_dict = pickle.load(f)
with open(data_path + 'Sisyphus_A_Ec_dict.pkl', 'rb') as f: Ec_dict = pickle.load(f)
with open(data_path + 'Sisyphus_A_Ej_dict.pkl', 'rb') as f: Ej_dict = pickle.load(f)


time: 0 ns (started: 2026-11-13 17:49:31 +01:00)


## Wafer drawing

In [4]:
# define layers
reso_layer = 1
patch_layer = 2
JJ_layer = 3
undercut_layer = 4
Al_1_demo = 5
Al_2_demo = 6


time: 0 ns (started: 2026-11-13 17:49:31 +01:00)


In [5]:
# metal GUI initialization
design = metal.designs.DesignPlanar()
gui = metal.MetalGUI(design)
design.overwrite_enabled = True


time: 1.62 s (started: 2026-11-13 17:49:31 +01:00)


In [49]:
# setup basic chip parameters
design._variables = {'cpw_width':'50um', 'cpw_gap':'10um', 'fudge_factor': 1/(1 - 0.04),}
chip_x = 2500
chip_y = 18500 # TEST (Claude): 18800 makes the chip box overlap the template solid wafer_edge by 150um -> HFSS error
design.chips.main.size.size_x = f'{chip_x}um'
design.chips.main.size.size_y = f'{chip_y}um'
design.chips.main.size.size_z = '-500um'
design.chips.main.size.center_y = '-1.75mm'
design.chips.main


{'material': 'silicon',
 'layer_start': '0',
 'layer_end': '2048',
 'size': {'center_x': '0.0mm',
  'center_y': '-1.75mm',
  'center_z': '0.0mm',
  'size_x': '2500um',
  'size_y': '18500um',
  'size_z': '-500um',
  'sample_holder_top': '890um',
  'sample_holder_bottom': '1650um'}}

time: 15 ms (started: 2026-11-13 18:25:50 +01:00)


In [7]:
design.delete_all_components()

time: 16 ms (started: 2026-11-13 17:49:33 +01:00)


### Transmon

In [8]:
# transmon
T_x = f'{0}um'
T_y = f'{5350}um'
pad_R_mm = 0.3
pad_gap_mm = 2.2

T = TransmonPocketTeethCustom(design, 'T',
                             options=dict(
                                pos_x = T_x,
                                pos_y = T_y,
                                orientation = '0',
                                pad_gap = f'{pad_gap_mm}mm',
                                pad_width_top = '0.mm',
                                pad_height_top = f'{pad_R_mm*3}mm',
                                edge_rad_top=f'{pad_R_mm}mm',
                                pad_width_bot = '0.mm',
                                pad_height_bot = f'{pad_R_mm*2}mm',
                                pocket_width = '2mm',
                                pocket_height = '4.1mm',
                                pocket_dy = '0mm',
                                layer=reso_layer,
                                layer_pocket= reso_layer,
                                draw_jj=False,
                                round_edges=True,
                               ))


time: 47 ms (started: 2026-11-13 17:49:33 +01:00)


In [9]:
gui.rebuild()

time: 46 ms (started: 2026-11-13 17:49:33 +01:00)


In [10]:
JJ_width = np.sqrt(Ej_dict['T']/exp_vars['Ej_to_S'])*1.2
taper_gap = 1
slope=0.1
evaporation_shift = exp_vars['evaporation_shift']
width_mod = evaporation_shift/np.sqrt(2)*(1 - slope)
undercut = exp_vars['undercut']
# length_cut_taper='14um'
length_cut_taper='53.75um'

# merging with pad circle
h = pad_R_mm/2*1e3
R = pad_R_mm*1e3
l = np.sqrt(h*(2*R - h))*2
k = (R - h)/np.sqrt(h*(2*R - h))


taper_j1 = Taper(design, 'taper_j1', options=dict(pos_x=T_x + f'-{width_mod/2}um',
                                                    pos_y=T_y+f'+{taper_gap/2}um',
                                                    orientation = '90',
                                                    width_in=f'{JJ_width*np.sqrt(2) + width_mod}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper+'+1.6um',
                                                    slope=slope,
                                                    end_slope=k,
                                                    draw='in',
                                                    undercut=f'{undercut}um',
                                                    layer=JJ_layer,
                                                    layer_undercut=undercut_layer,
                                                     holes = dict(
                                                        real_hole=dict(pos_x='10um+41.3um', pos_y=f'-{width_mod/2}um',
                                                                       width='2um', length='0.5um',
                                                                       fillet='0.1um', orientation='45'),
                                                        on_Nb_hole=dict(pos_x='14.6um+39.6um', pos_y='-0.6um'+f'-{width_mod/2}um',
                                                                       width=f'{0.5}um', length='5um',
                                                                       fillet='0.1um', orientation='0'),),))

taper_j1_shift_1 = Taper(design, 'taper_j1_shift_1', options=dict(pos_x=T_x + f'-{width_mod/2}um',
                                                    pos_y=T_y+f'+{taper_gap/2}um',
                                                    orientation = '90',
                                                    width_in=f'{JJ_width*np.sqrt(2) + width_mod}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper+'+1.6um',
                                                    slope=slope,
                                                    end_slope=k,
                                                    evaporation_angle='-45',
                                                    evaporation_shift=f'{evaporation_shift}um',
                                                    draw='in',
                                                    undercut=f'{undercut}um',
                                                    layer=Al_1_demo,
                                                     holes = dict(
                                                        real_hole=dict(pos_x='10um+41.3um', pos_y=f'-{width_mod/2}um',
                                                                       width='2um', length='0.5um',
                                                                       fillet='0.1um', orientation='45'),
                                                        on_Nb_hole=dict(pos_x='14.6um+39.6um', pos_y='-0.6um'+f'-{width_mod/2}um',
                                                                       width=f'{0.5}um', length='5um',
                                                                       fillet='0.1um', orientation='0'),),))


taper_j1_shift_2 = Taper(design, 'taper_j1_shift_2', options=dict(pos_x=T_x + f'-{width_mod/2}um',
                                                    pos_y=T_y+f'+{taper_gap/2}um',
                                                    orientation = '90',
                                                    width_in=f'{JJ_width*np.sqrt(2) + width_mod}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper+'+1.6um',
                                                    slope=slope,
                                                    end_slope=k,
                                                    evaporation_angle='+45',
                                                    evaporation_shift=f'{evaporation_shift}um',
                                                    draw='in',
                                                    undercut=f'{undercut}um',
                                                    layer=Al_2_demo,
                                                     holes = dict(
                                                        real_hole=dict(pos_x='10um+41.3um', pos_y=f'-{width_mod/2}um',
                                                                       width='2um', length='0.5um',
                                                                       fillet='0.1um', orientation='45'),
                                                        on_Nb_hole=dict(pos_x='14.6um+39.6um', pos_y='-0.6um'+f'-{width_mod/2}um',
                                                                       width=f'{0.5}um', length='5um',
                                                                       fillet='0.1um', orientation='0'),),))

taper_p1 = Taper(design, 'taper_p1', options=dict(pos_x=T_x,
                                                    pos_y=T_y+f'+{taper_gap/2}um',
                                                    orientation = '90',
                                                    width_in=f'{JJ_width*np.sqrt(2)}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper,
                                                    slope=slope,
                                                    end_slope=k,
                                                    draw='out',
                                                    layer=reso_layer,))





patch_w = 8
patch_1 = Patch(design, 'patch_1', options=dict(pos_x=T_x,
                                                    pos_y=T_y +'+13um+40um',
                                                     orientation = '90',
                                                     width=f'{patch_w}um',
                                                     height=f'{(1 + slope)*patch_w}um',
                                                     height_extra=f'{(1 - slope)*patch_w}um',
                                                     fillet='0.2um',
                                                     layer_patch=patch_layer,
                                                     holes=dict(
                                                         side=dict(pos_x=f'{-(patch_w/2 - (1 - slope)*patch_w/2 + 15/np.sqrt(2))}um', pos_y='0um',
                                                               width='40um', length='15um',
                                                               fillet='0um', orientation='45'),),
                                                    ),)


time: 94 ms (started: 2026-11-13 17:49:33 +01:00)


In [11]:
taper_j2 = Taper(design, 'taper_j2', options=dict(pos_x=T_x + f'-{width_mod/2}um',
                                                    pos_y=T_y+f'-{taper_gap/2}um',
                                                    orientation = '-90',
                                                    width_in=f'{JJ_width*np.sqrt(2) + width_mod}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper + '+1.6um',
                                                    slope=slope,
                                                    end_slope=k,
                                                    draw='in',
                                                    undercut=f'{undercut}um',
                                                    layer=JJ_layer,
                                                    layer_undercut=undercut_layer,
                                                     holes = dict(
                                                        real_hole=dict(pos_x='10um+41.3um', pos_y=f'+{width_mod/2}um',
                                                                       width='2um', length='0.5um',
                                                                       fillet='0.1um', orientation='45'),
                                                        on_Nb_hole=dict(pos_x='14.6um+39.6um', pos_y='-0.6um'+f'+{width_mod/2}um',
                                                                       width=f'{0.5}um', length='5um',
                                                                       fillet='0.1um', orientation='0'),),))

taper_j2_shift_1 = Taper(design, 'taper_j2_shift_1', options=dict(pos_x=T_x + f'-{width_mod/2}um',
                                                    pos_y=T_y+f'-{taper_gap/2}um',
                                                    orientation = '-90',
                                                    width_in=f'{JJ_width*np.sqrt(2) + width_mod}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper + '+1.6um',
                                                    slope=slope,
                                                    end_slope=k,
                                                    evaporation_angle='-45',
                                                    evaporation_shift=f'{evaporation_shift}um',
                                                    draw='in',
                                                    undercut=f'{undercut}um',
                                                    layer=Al_1_demo,
                                                     holes = dict(
                                                        real_hole=dict(pos_x='10um+41.3um', pos_y=f'+{width_mod/2}um',
                                                                       width='2um', length='0.5um',
                                                                       fillet='0.1um', orientation='45'),
                                                        on_Nb_hole=dict(pos_x='14.6um+39.6um', pos_y='-0.6um'+f'+{width_mod/2}um',
                                                                       width=f'{0.5}um', length='5um',
                                                                       fillet='0.1um', orientation='0'),),))

taper_j2_shift_2 = Taper(design, 'taper_j2_shift_2', options=dict(pos_x=T_x + f'-{width_mod/2}um',
                                                    pos_y=T_y+f'-{taper_gap/2}um',
                                                    orientation = '-90',
                                                    width_in=f'{JJ_width*np.sqrt(2) + width_mod}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper + '+1.6um',
                                                    slope=slope,
                                                    end_slope=k,
                                                    evaporation_angle='+45',
                                                    evaporation_shift=f'{evaporation_shift}um',
                                                    draw='in',
                                                    undercut=f'{undercut}um',
                                                    layer=Al_2_demo,
                                                     holes = dict(
                                                        real_hole=dict(pos_x='10um+41.3um', pos_y=f'+{width_mod/2}um',
                                                                       width='2um', length='0.5um',
                                                                       fillet='0.1um', orientation='45'),
                                                        on_Nb_hole=dict(pos_x='14.6um+39.6um', pos_y='-0.6um'+f'+{width_mod/2}um',
                                                                       width=f'{0.5}um', length='5um',
                                                                       fillet='0.1um', orientation='0'),),))

taper_p2 = Taper(design, 'taper_p2', options=dict(pos_x=T_x,
                                                    pos_y=T_y+f'-{taper_gap/2}um',
                                                    orientation = '-90',
                                                    width_in=f'{JJ_width*np.sqrt(2)}um',
                                                    width_out=f'{l}um',
                                                    length=f'{h + pad_gap_mm*1e3/2 - taper_gap/2}um',
                                                    length_cut=length_cut_taper,
                                                    slope=slope,
                                                    end_slope=k,
                                                    draw='out',
                                                    layer=reso_layer,))

patch_w = 8
patch_2 = Patch(design, 'patch_2', options=dict(pos_x=T_x,
                                                    pos_y=T_y +'-13um-40um',
                                                     orientation = '-90',
                                                     width=f'{patch_w}um',
                                                     height=f'{(1 + slope)*patch_w}um',
                                                     height_extra=f'{(1 - slope)*patch_w}um',
                                                     fillet='0.2um',
                                                     layer_patch=patch_layer,
                                                     holes=dict(
                                                         side=dict(pos_x=f'{-(patch_w/2 - (1 - slope)*patch_w/2 + 15/np.sqrt(2))}um', pos_y='0um',
                                                               width='40um', length='15um',
                                                               fillet='0um', orientation='45'),),
                                                    ),)


time: 78 ms (started: 2026-11-13 17:49:34 +01:00)


In [12]:
gui.rebuild()

time: 235 ms (started: 2026-11-13 17:49:34 +01:00)


In [13]:
jj = JunctionManhattan_cross(design, 'jj',
                           options=dict(pos_x=T_x + f'+{taper_gap/2}um',
                                        pos_y=T_y,
                                        orientation = '-45',
                                        length='2.3um',
                                        x_cross=f'{taper_gap/np.sqrt(2) + JJ_width/2}um',
                                        width=f'{JJ_width}um',
                                        layer=JJ_layer,
                                        undercut='20nm',
                                        undercut_big=f'{undercut}um',
                                        layer_undercut=undercut_layer,
                                        draw='all',
                                        helper='False'),
                          )


time: 16 ms (started: 2026-11-13 17:49:34 +01:00)


In [14]:
gui.rebuild()

time: 250 ms (started: 2026-11-13 17:49:34 +01:00)


### Read reso

In [15]:
read_reso_pad_A = OpenToGround(design, 'read_reso_pad_A', options=dict(pos_x=0, pos_y='2.05mm + 0.4mm', orientation = '90', width_in='cpw_width',))
read_reso_pad_B = OpenToGround(design, 'read_reso_pad_B', options=dict(pos_x=0, pos_y='0.67mm', orientation = '-90', width_in='cpw_width',))

read_reso = RouteMeander(design, 'read_reso',
                           options=dict(hfss_wire_bonds = False,
                                        pin_inputs=dict(
                                            start_pin=dict(component='read_reso_pad_A', pin='open',),
                                            end_pin=dict(component='read_reso_pad_B', pin='open',),
                                        ),
                                        fillet='100um',
                                        # subtract tee coupler segment out of transmon pocket
                                        total_length=f'{10.22}mm',
                                        meander=dict(spacing='0.25mm',
                                                     asymmetry='0um',
                                                     
                                                 ),
                                        lead=dict(start_straight='300um', end_straight='30um'),
                                        snap = 'True',
                                        layer=reso_layer,
                                        )
                           )


time: 15 ms (started: 2026-11-13 17:49:34 +01:00)


In [16]:
gui.rebuild()

time: 281 ms (started: 2026-11-13 17:49:34 +01:00)


### Purcell reso

In [17]:
R = 0.7#0.8
pin_y = -6.3
L = 2*R + 0.18
pos_y = pin_y - L + R

filter_reso_pad_A = CirclePad(design, 'filter_reso_pad_A', options=dict(pos_x=0, pos_y=f'{pos_y}mm', orientation = '90',
                                              width_in='cpw_width', merging='0.3', length=f'{L}mm', R=f'{R}mm', ))
filter_reso_pad_B = CirclePad(design, 'filter_reso_pad_B', options=dict(pos_x=0, pos_y=f'{pos_y-3+2*L-2*R}mm', orientation = '-90',
                                              width_in='cpw_width', merging='0.3', length=f'{L}mm', R=f'{R}mm', ))

filter_reso = RouteMeander(design, 'filter_reso',
                       options=dict(pin_inputs=dict(
                                        start_pin=dict(component='filter_reso_pad_A', pin='in',),
                                        end_pin=dict(component='filter_reso_pad_B', pin='in',),
                                    ),
                                    fillet='100um',
                                    # including experimental frequency shift
                                    total_length=f'{3.5}mm' + '+0.34mm',
                                    meander=dict(spacing='0.25mm',
                                                 asymmetry='0um',),
                                    lead=dict(start_straight='160um', end_straight='160um'),
                                    snap = 'True',
                                    layer=reso_layer,
                                    )
                       )


Non-monotonic tail derivative
Non-monotonic tail derivative
time: 16 ms (started: 2026-11-13 17:49:35 +01:00)


In [18]:
gui.rebuild()

Non-monotonic tail derivative
Non-monotonic tail derivative
time: 312 ms (started: 2026-11-13 17:49:35 +01:00)


# HFSS simulations

## Capacitance (Q3D)

In [30]:
q3d = Q3d(version=ansys_version, project='Sisyphus', design='Capacitance')

PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
time: 375 ms (started: 2026-11-13 18:20:54 +01:00)


In [31]:
fourq_q3d = design.renderers.q3d
fourq_q3d.start()
fourq_q3d.activate_ansys_design('Capacitance')


time: 328 ms (started: 2026-11-13 18:20:55 +01:00)


In [32]:
# add transmon to design
fourq_q3d.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)

# delete autoground
q3d.modeler.delete('ground_main_plane')

# TEST MODE (Claude): coarse convergence; original values are restored in the cleanup cell
q3d_setup = q3d.setups[0]
cap_orig = dict(q3d_setup.props['Cap'])
q3d_setup.props['Cap'].update(MaxPass=6, MinPass=2, MinConvPass=1, PerError=2)
q3d_setup.update()


True

time: 3.78 s (started: 2026-11-13 18:20:55 +01:00)


In [34]:
q3d.analyze('Setup')

True

time: 48.4 s (started: 2026-11-13 18:21:10 +01:00)


In [35]:
C_lom = fourq_q3d.get_capacitance_matrix()
display(C_lom[0])
print('unit:', C_lom[1])


,Cavity_V1,pad_bot_T,pad_top_T,Pin
Cavity_V1,3348.55153,-112.01428,-113.86894,-192.21881
pad_bot_T,-112.01428,146.42969,-34.35594,-0.05947
pad_top_T,-113.86894,-34.35594,148.23158,-0.00482
Pin,-192.21881,-0.05947,-0.00482,192.28310


unit: fF
time: 78 ms (started: 2026-11-13 18:21:58 +01:00)


In [36]:
# order island names
island_sequence = []
island_sequence.append('pad_bot_T -> T_bot')
island_sequence.append('pad_top_T -> T_top')

# order pins names
pins = []
pins.append('Pin -> pin')

# conver C_lom to fancy form
C_df, C_pins_df = tul.lom_c_matrix_convert(C_lom[0], island_sequence, pins=pins, physical=False)

display(C_df)
display(C_pins_df)

C = np.asarray(C_df)
C_pins = np.asarray(C_pins_df)

# add jj capacitance
C[0, 1] += tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])


,0: T_bot,1: T_top
0: T_bot,112.07375,34.35594
1: T_top,0.00000,113.87564


,0: pin
0: T_bot,0.05947
1: T_top,0.00482


time: 16 ms (started: 2026-11-13 18:21:58 +01:00)


In [37]:
# transfer matrix
S = np.eye(C.shape[0])

# first mode will be amtisym = transmon
S[0, 1] = -1
S[1, 0] = 1

Ec, pin_q = tul.legendre_forward(C, C_pins=C_pins, S=S)
print(f'Transmon Ec = {np.around(Ec[0, 0], 3)} GHz')
s, _, n = tul.transmon(Ej_dict['T'], Ec[0, 0], truncation=20)
print(f'f01: {np.around(s[1], 3)} GHz\nalpha: {np.around(s[2] - 2*s[1], 3)} GHz')
print(f'pin_q: {np.around(pin_q[0, 0], 3)} GHz/mV')

Ec_dict.update({'Ec_T':Ec})
Ec_dict.update({'pin_q':pin_q})
Ec_dict.update({'C':C})
Ec_dict.update({'C_pins':C_pins})

f_dict.update({'T':s[1]})
f_dict.update({'alpha':s[2] - 2*s[1]})


Transmon Ec = 0.207 GHz
f01: 6.09 GHz
alpha: -0.225 GHz
pin_q: 0.143 GHz/mV
time: 0 ns (started: 2026-11-13 18:21:58 +01:00)


In [38]:
# delete old qiskit design
q3d.modeler.delete(q3d.modeler.get_objects_in_group('Sheets'))
q3d.modeler.delete('main')

# TEST MODE (Claude): restore the original convergence settings of the template setup
q3d_setup.props['Cap'].update(cap_orig)
q3d_setup.update()

# break connection with ANSYS
fourq_q3d.stop()
q3d.release_desktop(close_projects=False, close_desktop=False)


True

time: 859 ms (started: 2026-11-13 18:21:59 +01:00)


In [39]:
# save
with open(data_path + 'Sisyphus_A_Ec_dict.pkl', 'wb') as f: pickle.dump(Ec_dict, f)
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'wb') as f: pickle.dump(f_dict, f)


time: 15 ms (started: 2026-11-13 18:21:59 +01:00)


## Eigenmode

### Bare readout reso

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Eigenmode')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Eigenmode')


In [ ]:
# add read reso and transmon
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 'read_reso_pad_A', 'read_reso_pad_B'], 
                         [], box_plus_buffer=False)

# delete autoground and fake jj
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3

rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="50um", 
                             name='transmon_mesh', maximum_elements=None,)

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_read_reso'], inside_selection=False, maximum_length="10um", 
                             name='meander_mesh', maximum_elements=None,)

# setup config
setup = hfss.setups[0]
setup.props["MinimumFrequency"] = "7.5GHz"   
setup.props["NumModes"] = 1
setup.update()


In [ ]:
hfss.analyze('Setup')

In [ ]:
sol = hfss.post.get_solution_data(
    expressions=[f're(Mode({n+1}))' for n in range(1)]+[f'Q({n+1})' for n in range(1)],
    setup_sweep_name=f"{hfss.setups[0].name} : LastAdaptive",
    report_category="Eigenmode Parameters",
)

freqs = []
Qs = []

for n in range(1):
    freqs.append(sol.data_real(f"re(Mode({n+1}))"))
    Qs.append(sol.data_real(f"Q({n+1})"))

freqs = np.around(np.asarray(freqs)/1e9, 3)
Qs = np.around(np.asarray(Qs))

display(pd.DataFrame([freqs[:, 0], Qs[:, 0]], index=['Q', 'Frequency, GHz'], columns=['read']))


In [ ]:
f_dict.update({'read_bare':freqs[0, 0]})
Q_dict.update({'read_bare':Qs[0, 0]})


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Bare filter reso

In [50]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Eigenmode')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Eigenmode')


PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
time: 718 ms (started: 2026-11-13 18:25:50 +01:00)


In [51]:
# add filter design
fourq_hfss.render_design(['filter_reso', 'filter_reso_pad_A', 'filter_reso_pad_B'], 
                         [], box_plus_buffer=False)

# delete autoground and fake jj
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_filter_reso'], inside_selection=False, maximum_length="100um", #"5um", 
                             name='meander_mesh', maximum_elements=None,)

# setup config
setup = hfss.setups[0]
setup.props["MinimumFrequency"] = "7.5GHz"   
setup.props["NumModes"] = 1
setup.update()

# TEST MODE (Claude): coarse convergence; original values are restored in the cleanup cell
eig_orig = {k: setup.props[k] for k in ('MaximumPasses', 'MinimumPasses', 'MinimumConvergedPasses')}
setup.props.update(MaximumPasses=4, MinimumPasses=2, MinimumConvergedPasses=1)
setup.update()


True

time: 5.19 s (started: 2026-11-13 18:25:51 +01:00)


In [53]:
hfss.analyze('Setup')

True

time: 17.2 s (started: 2026-11-13 18:26:06 +01:00)


In [54]:
sol = hfss.post.get_solution_data(
    expressions=[f're(Mode({n+1}))' for n in range(1)]+[f'Q({n+1})' for n in range(1)],
    setup_sweep_name=f"{hfss.setups[0].name} : LastAdaptive",
    report_category="Eigenmode Parameters",
)

freqs = []
Qs = []

for n in range(1):
    freqs.append(sol.data_real(f"re(Mode({n+1}))"))
    Qs.append(sol.data_real(f"Q({n+1})"))

freqs = np.around(np.asarray(freqs)/1e9, 3)
Qs = np.around(np.asarray(Qs))

display(pd.DataFrame([freqs[:, 0], Qs[:, 0]], index=['Frequency, GHz', 'Q'], columns=['filter']))


Time to initialize solution data:0.0040035247802734375
Time to initialize solution data:0.007007598876953125


,filter
"Frequency, GHz",8.261
Q,137.000


time: 390 ms (started: 2026-11-13 18:26:24 +01:00)


In [55]:
f_dict.update({'filter_bare' : freqs[0, 0]})
Q_dict.update({'filter_bare':Qs[0, 0]})


time: 16 ms (started: 2026-11-13 18:26:24 +01:00)


In [56]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# TEST MODE (Claude): restore the original convergence settings of the template setup
setup.props.update(eig_orig)
setup.update()

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


PyAEDT WARNING: No objects to delete


True

time: 844 ms (started: 2026-11-13 18:26:24 +01:00)


#### Pin variation

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Eigenmode')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Eigenmode')


In [ ]:
# add filter design
fourq_hfss.render_design(['filter_reso', 'filter_reso_pad_A', 'filter_reso_pad_B'], 
                         [], box_plus_buffer=False)

# delete autoground and fake jj
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_filter_reso'], inside_selection=False, maximum_length="25um", 
                             name='meander_mesh', maximum_elements=None,)
 
# setup config
setup = hfss.setups[0]
setup.props["MinimumFrequency"] = "7.5GHz"   
setup.props["NumModes"] = 1
setup.update()


In [ ]:
# define y shift
hfss.variable_manager.set_variable(name="drift", expression=f"{+0.5}mm", overwrite=True)

In [ ]:
print(hfss.setup_names)                    # solve setups actually present
print(hfss.ooptimetrics.GetSetupNames())   # parametric setups already present (stale ones too)
print(hfss["drift"])                       # value and units of the swept variable

In [ ]:
par = hfss.parametrics.add("drift", "-0.5mm", "0.5mm", "0.1mm", "LinearStep", solution="Setup", name="drift_sweep")

In [ ]:
par.props["ProdOptiSetupDataV2"]["SaveFields"] = False
par.update()

hfss.save_project()
hfss.analyze_setup("drift_sweep", cores=8, num_variations_to_distribute=2)

In [ ]:
mode = 1
sol = hfss.post.get_solution_data(
    expressions=[f"re(Mode({mode}))", f"Q({mode})"],
    setup_sweep_name=f"{hfss.setups[0].name} : LastAdaptive",
    report_category="Eigenmode Parameters",
    primary_sweep_variable="drift",          # drift becomes the x-axis of the report
    variations={"drift": ["All"]},           # include every solved variation, not just the nominal one
)

drift = np.asarray(sol.primary_sweep_values, dtype=float)                 # in sol.units_sweeps["drift"]
freqs = np.around(np.asarray(sol.data_real(f"re(Mode({mode}))")) / 1e9, 3)   # GHz
Qs    = np.around(np.asarray(sol.data_real(f"Q({mode})")))

print(sol.units_sweeps["drift"], sol.units_data[f"re(Mode({mode}))"])   # expect mm and Hz


In [ ]:
# results processing and saving
kappas = freqs/Qs

plt.figure(figsize=(8, 6))
plt.plot(drift, freqs, color='blue', alpha=0.5)
plt.xlabel('Y-axis shift in direction of cavity, mm')
plt.ylabel('filter frequency, GHz')
plt.title('Eigenmode Y-sweep simulation') 
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(drift, kappas*1e3, color='blue', alpha=0.5)
plt.xlabel('Y-axis shift in direction of cavity, mm')
plt.ylabel('filter kappa, MHz')
plt.title('Eigenmode Y-sweep simulation') 
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

np.savez(data_path+'filter_y_shift_eigenmode', drift=drift, freqs=freqs, kappas=kappas)


In [ ]:
# define y shift
hfss.variable_manager.set_variable(name="drift", expression=f"{+0.}mm", overwrite=True)

In [ ]:
print(hfss.setup_names)                    # solve setups actually present
print(hfss.ooptimetrics.GetSetupNames())   # parametric setups already present (stale ones too)
print(hfss["drift"])                       # value and units of the swept variable

In [ ]:
par = hfss.parametrics.add("hhh", "-2.4mm", "-2mm", "0.05mm", "LinearStep", solution="Setup", name="heigh_sweep")

In [ ]:
par.props["ProdOptiSetupDataV2"]["SaveFields"] = False
par.update()

hfss.save_project()
hfss.analyze_setup("heigh_sweep", cores=8, num_variations_to_distribute=2)


In [ ]:
mode = 1
sol = hfss.post.get_solution_data(
    expressions=[f"re(Mode({mode}))", f"Q({mode})"],
    setup_sweep_name=f"{hfss.setups[0].name} : LastAdaptive",
    report_category="Eigenmode Parameters",
    primary_sweep_variable="hhh",          # drift becomes the x-axis of the report
    variations={"hhh": ["All"]},           # include every solved variation, not just the nominal one
)

h = np.asarray(sol.primary_sweep_values, dtype=float)                 # in sol.units_sweeps["drift"]
freqs = np.around(np.asarray(sol.data_real(f"re(Mode({mode}))")) / 1e9, 3)   # GHz
Qs    = np.around(np.asarray(sol.data_real(f"Q({mode})")))

print(sol.units_sweeps["hhh"], sol.units_data[f"re(Mode({mode}))"])   # expect mm and Hz


In [ ]:
# results processing and saving
kappas = freqs/Qs

plt.figure(figsize=(4, 3))
plt.plot(h, freqs, color='blue', alpha=0.5)
plt.xlabel('Z-axis shift of pin, mm')
plt.ylabel('filter frequency, GHz')
plt.title('Eigenmode z-sweep simulation') 
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(4, 3))
plt.plot(h, kappas*1e3, color='blue', alpha=0.5)
plt.xlabel('Z-axis shift of pin, mm')
plt.ylabel('filter kappa, MHz')
plt.title('Eigenmode Z-sweep simulation') 
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

np.savez(data_path+'filter_z_shift_eigenmode', drift=drift, freqs=freqs, kappas=kappas)


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Complete

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Eigenmode')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Eigenmode')


In [ ]:
# add full design
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 'read_reso_pad_A', 'read_reso_pad_B',
                          'filter_reso', 'filter_reso_pad_A', 'filter_reso_pad_B'], 
                         [], box_plus_buffer=False)

# delete autoground and fake jj
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3

rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_read_reso', 'trace_filter_reso'], inside_selection=False, maximum_length="10um", 
                             name='meander_mesh', maximum_elements=None,)

# setup config
setup = hfss.setups[0]
setup.props["MinimumFrequency"] = "3GHz"   
setup.props["NumModes"] = 5
setup.update()


In [ ]:
hfss.analyze('Setup')

In [ ]:
sol = hfss.post.get_solution_data(
    expressions=[f're(Mode({n+1}))' for n in range(5)]+[f'Q({n+1})' for n in range(5)],
    setup_sweep_name=f"{hfss.setups[0].name} : LastAdaptive",
    report_category="Eigenmode Parameters",
)

freqs = []
Qs = []

for n in range(5):
    freqs.append(sol.data_real(f"re(Mode({n+1}))"))
    Qs.append(sol.data_real(f"Q({n+1})"))

freqs = np.around(np.asarray(freqs)/1e9, 3)
Qs = np.around(np.asarray(Qs))

display(pd.DataFrame([freqs[:, 0], Qs[:, 0]], index=['Frequency, GHz', 'Q'], columns=['storage', 'transmon', 'filter', 'read', 'high-post']))

f_dict.update({'storage':freqs[0, 0]})
f_dict.update({'T_egm':freqs[1, 0]})
f_dict.update({'read':freqs[1, 0]})
f_dict.update({'filter':freqs[1, 0]})
f_dict.update({'high-post':freqs[4, 0]})

Q_dict.update({'storage_egm':Qs[0, 0]})
Q_dict.update({'T_egm':Qs[1, 0]})
Q_dict.update({'read_egm':Qs[2, 0]})
Q_dict.update({'filter_egm':Qs[3, 0]})
Q_dict.update({'high-post_egm':Qs[4, 0]})


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Save

In [ ]:
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'wb') as f: pickle.dump(f_dict, f)
with open(data_path + 'Sisyphus_A_Q_dict.pkl', 'wb') as f: pickle.dump(Q_dict, f)


## Qubit BBQ (Y-port at qubit)

In [64]:
def qubit_bbq_fit(freqs, Y, L_list, tolY=1e-8):
    # resonant frequencies search
    L_list_clean = []
    reso_list=[]
    for n in range(L_list.shape[0]):
        tmp = tul.bbq_resonances_of_Y(np.asarray(freqs, dtype=float), Y, L=L_list[n], 
                                      C=tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S']), tolY=tolY)
        if tmp.shape[0] == 2:  
            reso_list.append(tmp)
            L_list_clean.append(L_list[n])
    
    reso_list = np.asarray(reso_list)
    L_list = np.asarray(L_list_clean)

    # fit f(L) curves
    def loss(x):
    
        f = x[0]
        C_T = x[1]
        g = x[2]
        L0_T = x[3]
       
        error = 0        
        for n in range(L_list.shape[0]):
            f_T =  1/(2*np.pi)/np.sqrt((L_list[n]+L0_T)*C_T)*1e3
            Z_T = np.sqrt((L_list[n]+L0_T)*1e6/C_T)
            reso_predict, _, _ = tul.mix_oscillators([f, f_T], g*tul.n_zpf(Z_T))
            error += np.linalg.norm(reso_list[n, :] - reso_predict)
        return error
            
    sol = scp.optimize.dual_annealing(loss, bounds=[(2, 10), (20, 200), (0, 0.4), (0, 8)], maxiter=1000)
    fit_list = []
    f = sol.x[0]
    C_T = sol.x[1]
    g = sol.x[2]
    L0_T = sol.x[3]
    
    for n in range(L_list.shape[0]):
        f_T =  1/(2*np.pi)/np.sqrt((L_list[n]+L0_T)*C_T)*1e3
        Z_T = np.sqrt((L_list[n]+L0_T)*1e6/C_T)
        reso_predict, _, _ = tul.mix_oscillators([f, f_T], g*tul.n_zpf(Z_T))
        fit_list.append(reso_predict)

    fit_list = np.asarray(fit_list)
    
    return L_list, reso_list, fit_list, f, C_T, g, L0_T
    

time: 0 ns (started: 2026-11-13 18:28:11 +01:00)


### To storage

In [59]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
time: 469 ms (started: 2026-11-13 18:27:15 +01:00)


In [62]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add JJ
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=1, name='JJ_port')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="100um", #"30um", 
                             name='transmon_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['storage']}GHz')
setup.update_property('Passes', '4') # '20'   TEST (Claude): coarse
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{3.8}GHz', RangeEnd=f'{5.2}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


PyAEDT WARNING: Parameter 'ID' is invalid and will be ignored.


True

time: 4.08 s (started: 2026-11-13 18:28:03 +01:00)


In [65]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

True

time: 45.3 s (started: 2026-11-13 18:28:11 +01:00)


In [66]:
data = hfss.post.get_solution_data(expressions=['re(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


PyAEDT WARNING: No report category provided. Automatically identified Modal Solution Data
Time to initialize solution data:0.01691579818725586
Time to initialize solution data:0.07383155822753906
time: 2.22 s (started: 2026-11-13 18:28:57 +01:00)


In [67]:
L_list, reso_list, fit_list, f_storage, C_T, g, L0_T = qubit_bbq_fit(freqs, Y, L_list=np.linspace(15, 11, 101))

time: 1min 32s (started: 2026-11-13 18:28:59 +01:00)


In [68]:
print(f'storage frequency: {np.around(f_storage, 3)} GHz')
print(f'transmon: C={np.around(C_T, 3)} fF, Ec={np.around(tul.Ec_of_C(C_T), 3)} GHz, L0={np.around(L0_T, 3)} nH')
print(f'g: {np.around(g*1e3, 3)} MHz\n')

f_dict.update({'storage_bbq':f_storage})
Ej_dict.update({'L0':L0_T})
Ec_dict.update({'C_bbq':C_T})
g_dict.update({'T-stor':g})


storage frequency: 4.444 GHz
transmon: C=92.324 fF, Ec=0.21 GHz, L0=1.008 nH
g: 18.667 MHz

time: 0 ns (started: 2026-11-13 18:30:31 +01:00)


In [69]:
plt.figure(figsize=(8, 6))
plt.plot(L_list, reso_list[:, 0], label='simulation', color='blue', alpha=0.5)
plt.plot(L_list, reso_list[:, 1], color='blue', alpha=0.5)
plt.plot(L_list, fit_list[:, 0], label='fit', color='black', linestyle='dashed')
plt.plot(L_list, fit_list[:, 1], color='black', linestyle='dashed')

plt.xlabel('L, nH')
plt.ylabel('f, GHz')
plt.title('Transmon-Storage BBQ') 
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


time: 109 ms (started: 2026-11-13 18:30:31 +01:00)


In [73]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


True

time: 1.5 s (started: 2026-11-13 18:32:15 +01:00)


### To readout

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 
                          'read_reso_pad_A', 'read_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add JJ
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=1, name='JJ_port')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_read_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['read_bare']}GHz')
setup.update_property('Passes', '6')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{7.7}GHz', RangeEnd=f'{8.9}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=2000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
L_list, reso_list, fit_list, f_read, C_T, g, L0_T = qubit_bbq_fit(freqs, Y, L_list=np.linspace(2.2, 3, 101), tolY=1e-6)

In [ ]:
print(f'readout frequency: {np.around(f_read, 3)} GHz')
print(f'transmon: C={np.around(C_T, 3)} fF, Ec={np.around(tul.Ec_of_C(C_T), 3)} GHz, L0={np.around(L0_T, 3)} nH')
print(f'g: {np.around(g*1e3, 3)} MHz\n')

f_dict.update({'read_bbq':f_read})
Ej_dict.update({'L0_r':L0_T})
Ec_dict.update({'C_bbq_r':C_T})
g_dict.update({'T-read':g})


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(L_list, reso_list[:, 0], label='simulation', color='blue', alpha=0.5)
plt.plot(L_list, reso_list[:, 1], color='blue', alpha=0.5)
plt.plot(L_list, fit_list[:, 0], label='fit', color='black', linestyle='dashed')
plt.plot(L_list, fit_list[:, 1], color='black', linestyle='dashed')

plt.xlabel('L, nH')
plt.ylabel('f, GHz')
plt.title('Transmon-Readout BBQ') 
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### To filter

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'filter_reso', 
                          'filter_reso_pad_A', 'filter_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add JJ
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=1, name='JJ_port')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="60um", #"30um", 
                             name='transmon_mesh', maximum_elements=None,)

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_filter_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['filter_bare']}GHz')
setup.update_property('Passes', '20')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{7.9}GHz', RangeEnd=f'{8.3}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
L_list, reso_list, fit_list, f_filt, C_T, g, L0_T = qubit_bbq_fit(freqs, Y, L_list=np.linspace(2.8, 3.5, 301))

In [ ]:
print(f'filter frequency: {np.around(f_filt, 3)} GHz')
print(f'transmon: C={np.around(C_T, 3)} fF, Ec={np.around(tul.Ec_of_C(C_T), 3)} GHz, L0={np.around(L0_T, 3)} nH')
print(f'g: {np.around(g*1e3, 3)} MHz\n')

f_dict.update({'filt_bbq':f_filt})
Ej_dict.update({'L0_f':L0_T})
Ec_dict.update({'C_bbq_f':C_T})
g_dict.update({'T-filt':g})


In [ ]:
print(f'filter frequency: {np.around(f_filt, 3)} GHz')
print(f'transmon: C={np.around(C_T, 3)} fF, Ec={np.around(tul.Ec_of_C(C_T), 3)} GHz, L0={np.around(L0_T, 3)} nH')
print(f'g: {np.around(g*1e3, 3)} MHz\n')

f_dict.update({'filt_bbq':f_filt})
Ej_dict.update({'L0_f':L0_T})
Ec_dict.update({'C_bbq_f':C_T})
g_dict.update({'T-filt':g})


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(L_list, reso_list[:, 0], label='simulation', color='blue', alpha=0.5)
plt.plot(L_list, reso_list[:, 1], color='blue', alpha=0.5)
plt.plot(L_list, fit_list[:, 0], label='fit', color='black', linestyle='dashed')
plt.plot(L_list, fit_list[:, 1], color='black', linestyle='dashed')

plt.xlabel('L, nH')
plt.ylabel('f, GHz')
plt.title('Transmon-Filter BBQ') 
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Save

In [74]:
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'wb') as f: pickle.dump(f_dict, f)
with open(data_path + 'Sisyphus_A_Ec_dict.pkl', 'wb') as f: pickle.dump(Ec_dict, f)
with open(data_path + 'Sisyphus_A_g_dict.pkl', 'wb') as f: pickle.dump(g_dict, f)
with open(data_path + 'Sisyphus_A_Ej_dict.pkl', 'wb') as f: pickle.dump(Ej_dict, f)


time: 0 ns (started: 2026-11-13 18:32:17 +01:00)


## Reso BBQ (Y-port at read reso)

In [ ]:
def reso_bbq_fit(freqs, Y, L_list):
    # resonant frequencies search
    L_list_clean = []
    reso_list=[]
    for n in range(L_list.shape[0]):
        tmp = tul.bbq_resonances_of_Y(np.asarray(freqs, dtype=float), Y, L=L_list[n], C=0)
        if tmp.shape[0] == 2:  
            reso_list.append(tmp)
            L_list_clean.append(L_list[n])
    
    reso_list = np.asarray(reso_list)
    L_list = np.asarray(L_list_clean)

    # fit f(L) curves
    def loss(x):

        # f_reso(L) Tayloring
        f1 = x[0]
        f1_to_L = x[1]
        f1_to_L2 = x[2]
        
        f2 = x[3]
        g = x[4]
        
        error = 0        
        for n in range(L_list.shape[0]):
            # f_reso(L) Tayloring
            L0 = (L_list[0] + L_list[-1])/2
            f = f1 + f1_to_L*(L_list[n] - L0) + f1_to_L2*(L_list[n] - L0)**2
            reso_predict, _, _ = tul.mix_oscillators([f, f2], g)
            error += np.linalg.norm(reso_list[n, :] - reso_predict)
        return error
            
    sol = scp.optimize.dual_annealing(loss, bounds=[(2, 9), (-2, 0), (-0.5, +0.5), (4, 9), (0, 0.2)], maxiter=1000)
    f1 = sol.x[0]
    f1_to_L = sol.x[1]
    f1_to_L2 = sol.x[2]
    
    f2 = sol.x[3]
    g = sol.x[4]

    fit_list=[]
    for n in range(L_list.shape[0]):
        # f_reso(L) Tayloring
        L0 = (L_list[0] + L_list[-1])/2
        f = f1 + f1_to_L*(L_list[n] - L0) + f1_to_L2*(L_list[n] - L0)**2         
        reso_predict, _, _ = tul.mix_oscillators([f, f2], g)
        fit_list.append(reso_predict)

    fit_list = np.asarray(fit_list)
    
    return L_list, reso_list, fit_list, f1, f1_to_L, f1_to_L2, f2, g
    

### To storage

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 
                          'read_reso_pad_A', 'read_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add transmon JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3

rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ_T',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ_T', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

# add reso JJ
meander = hfss.modeler['trace_read_reso']

# find proper port position
n = len(meander.vertex_positions)//8 //2
coords = np.asarray(meander.vertex_positions[n*8 + 4:n*8+8])

# building proper rectangle
lx = max(np.abs(coords[0, 1] - coords[1, 1])*0.1, 0.1)
ly = np.abs(coords[0, 1] - coords[1, 1])

rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=np.average(coords, axis=0) - np.asarray([lx/2, ly/2, 0]),
                                     sizes = [f'{lx}mm', 
                                              f'{ly}mm'],
                                     name='JJ_hole',
                                     is_covered=True,
                                    )

# remove `tool_obj` from `blank_obj`; blank_obj is what remains
hfss.modeler.subtract(meander, 'JJ_hole', keep_originals=False)

rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=np.average(coords, axis=0) - np.asarray([lx/2, ly/2, 0]),
                                     sizes = [f'{lx}mm', 
                                              f'{ly}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=0, name='JJ_port')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ_T', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['JJ', 'trace_read_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['storage']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{4.3}GHz', RangeEnd=f'{4.6}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
L_list, reso_list, fit_list, f1, f1_to_L, f1_to_L2, f2, g = reso_bbq_fit(freqs, Y, L_list=np.linspace(4.8, 5.6, 101))

In [ ]:
print(f'cavity frequency: {np.around(f2, 3)} GHz')
print(f'g: {np.around(g*1e3, 3)} MHz\n')

g_dict.update({'read-stor_via_T':g})


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(L_list, reso_list[:, 0], label='simulation', color='blue', alpha=0.5)
plt.plot(L_list, reso_list[:, 1], color='blue', alpha=0.5)
plt.plot(L_list, fit_list[:, 0], label='fit', color='black', linestyle='dashed')
plt.plot(L_list, fit_list[:, 1], color='black', linestyle='dashed')

plt.xlabel('L, nH')
plt.ylabel('f, GHz')
plt.title('Readout-Storage BBQ') 
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')
hfss.modeler.delete('JJ_T')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### To filter

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['filter_reso', 'filter_reso_pad_A', 'filter_reso_pad_B',
                          'read_reso', 'read_reso_pad_A', 'read_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add reso JJ
meander = hfss.modeler['trace_read_reso']

# find proper port position
n = len(meander.vertex_positions)//8 //2
coords = np.asarray(meander.vertex_positions[n*8 + 4:n*8+8])

# building proper rectangle
lx = max(np.abs(coords[0, 1] - coords[1, 1])*0.1, 0.1)
ly = np.abs(coords[0, 1] - coords[1, 1])

rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=np.average(coords, axis=0) - np.asarray([lx/2, ly/2, 0]),
                                     sizes = [f'{lx}mm', 
                                              f'{ly}mm'],
                                     name='JJ_hole',
                                     is_covered=True,
                                    )

# remove `tool_obj` from `blank_obj`; blank_obj is what remains
hfss.modeler.subtract(meander, 'JJ_hole', keep_originals=False)

rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=np.average(coords, axis=0) - np.asarray([lx/2, ly/2, 0]),
                                     sizes = [f'{lx}mm', 
                                              f'{ly}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=0, name='JJ_port')

#  assign the meander mesh
hfss.mesh.assign_length_mesh(['JJ', 'trace_read_reso', 'trace_filter_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['storage']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{8}GHz', RangeEnd=f'{9}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
L_list, reso_list, fit_list, f1, f1_to_L, f1_to_L3, f2, g = reso_bbq_fit(freqs, Y, L_list=np.linspace(-0.15, 0.3, 101))

In [ ]:
print(f'filter frequency: {np.around(f2, 3)} GHz')
print(f'g: {np.around(g*1e3, 3)} MHz\n')

f_dict.update({'filt_bbq_r': f2})
g_dict.update({'read-filter':g})


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(L_list, reso_list[:, 0], label='simulation', color='blue', alpha=0.5)
plt.plot(L_list, reso_list[:, 1], color='blue', alpha=0.5)
plt.plot(L_list, fit_list[:, 0], label='fit', color='black', linestyle='dashed')
plt.plot(L_list, fit_list[:, 1], color='black', linestyle='dashed')

plt.xlabel('L, nH')
plt.ylabel('f, GHz')
plt.title('Readout-Filter BBQ') 
plt.grid(True, alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')
# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Save

In [ ]:
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'wb') as f: pickle.dump(f_dict, f)
with open(data_path + 'Sisyphus_A_g_dict.pkl', 'wb') as f: pickle.dump(g_dict, f)


## Pin S-parameter

### Filter

In [75]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


PyAEDT WARNING: Service Pack is not detected. PyAEDT is currently connecting in Insecure Mode.
PyAEDT WARNING: Please download and install latest Service Pack to use connect to AEDT in Secure Mode.
time: 656 ms (started: 2026-11-13 18:32:17 +01:00)


In [76]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'filter_reso', 
                          'filter_reso_pad_A', 'filter_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="100um", #"30um", 
                             name='transmon_mesh', maximum_elements=None,)


#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_filter_reso'], inside_selection=False, maximum_length="100um", #"30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['filter_bare']}GHz')
setup.update_property('Passes', '5') # '20'   TEST (Claude): coarse
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{7.6}GHz', RangeEnd=f'{9}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


PyAEDT WARNING: Parameter 'ID' is invalid and will be ignored.


True

time: 8.56 s (started: 2026-11-13 18:32:17 +01:00)


In [78]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

True

time: 1min 32s (started: 2026-11-13 18:32:30 +01:00)


In [79]:
data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))'], primary_sweep_variable="Freq")
S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


PyAEDT WARNING: No report category provided. Automatically identified Modal Solution Data
Time to initialize solution data:0.01645183563232422
Time to initialize solution data:0.06501531600952148
time: 828 ms (started: 2026-11-13 18:34:02 +01:00)


In [80]:
# fit
pin = rfit.reflection_port()
pin.add_data(freqs*1e9, S)
pin.autofit(electric_delay=0)

pin.plotall()
display(pd.DataFrame([pin.fitresults]).applymap(lambda x: "{0:.2e}".format(x)))
print(f'bandwidth: {np.around(pin.fitresults['fr']/pin.fitresults['Ql']*1e-6, 3)} MHz\n')

f_dict.update({'filter_via_pin_S':pin.fitresults['fr']/1e9})
Qe_dict.update({'filter_via_pin_S':pin.fitresults['Qc']})
Qi_dict.update({'filter_via_pin_S':pin.fitresults['Qi']})


,Qi,Qc,Ql,fr,theta0,Ql_err,Qc_err,fr_err,chi_square,Qi_err
0,8.87e+04,1.30e+02,1.30e+02,8.18e+09,-1.64e-04,1.99e-01,1.45e-01,3.34e+04,1.89e-03,6.41e+04


bandwidth: 63.121 MHz

time: 532 ms (started: 2026-11-13 18:34:03 +01:00)


In [82]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# TEST MODE (Claude): put the template setup back as it was found on 2026-11-13
setup.update_property('Solution Freq', '8.344GHz')
setup.update_property('Passes', '20')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart='7.9GHz', RangeEnd='8.3GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


True

time: 1.52 s (started: 2026-11-13 18:34:28 +01:00)


### Readout

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 
                          'read_reso_pad_A', 'read_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)


#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_read_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['filter_bare']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{8.34}GHz', RangeEnd=f'{8.4}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))'], primary_sweep_variable="Freq")
S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
# fit
pin = rfit.reflection_port()
pin.add_data(freqs*1e9, S)
pin.autofit(electric_delay=0)

pin.plotall()
display(pd.DataFrame([pin.fitresults]).applymap(lambda x: "{0:.2e}".format(x)))
print(f'bandwidth: {np.around(pin.fitresults['fr']/pin.fitresults['Ql']*1e-6, 4)} MHz\n')

f_dict.update({'read_via_pin_S':pin.fitresults['fr']/1e9})
Qe_dict.update({'read_via_pin_S':pin.fitresults['Qc']})
Qi_dict.update({'read_via_pin_S':pin.fitresults['Qi']})


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Transmon (cannot see)

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 
                          'read_reso_pad_A', 'read_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)


#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_read_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['T_egm']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{f_dict['T_egm']-0.15}GHz', RangeEnd=f'{f_dict['T_egm']+0.15}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))'], primary_sweep_variable="Freq")
S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
# fit
pin = rfit.reflection_port()
pin.add_data(freqs*1e9, S)
pin.autofit(electric_delay=0)

pin.plotall()
display(pd.DataFrame([pin.fitresults]).applymap(lambda x: "{0:.2e}".format(x)))
print(f'bandwidth: {np.around(pin.fitresults['fr']/pin.fitresults['Ql']*1e-6, 4)} MHz\n')

f_dict.update({'T_via_pin_S':pin.fitresults['fr']})
Qe_dict.update({'T_via_pin_S':pin.fitresults['Qc']})
Qi_dict.update({'T_via_pin_S':pin.fitresults['Qi']})


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Complete

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Network')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Network')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2', 'read_reso', 'read_reso_pad_A', 'read_reso_pad_B', 
                          'filter_reso', 'filter_reso_pad_A', 'filter_reso_pad_B'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)


#  assign the meander mesh
hfss.mesh.assign_length_mesh(['trace_filter_reso', 'trace_read_reso'], inside_selection=False, maximum_length="30um", 
                             name='meander_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['filter_bare']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{7.6}GHz', RangeEnd=f'{9}GHz', RangeCount=24001, InterpTolerance=0.1, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))'], primary_sweep_variable="Freq")
S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(freqs, np.abs(S), color='blue')

plt.xlabel('f, GHz')
plt.ylabel('S')
plt.title('S-parameter scan') 
plt.grid(True, alpha=0.3)
plt.legend()
plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Save

In [83]:
with open(data_path + 'Sisyphus_A_Qi_dict.pkl', 'wb') as f: pickle.dump(Qi_dict, f)
with open(data_path + 'Sisyphus_A_Qe_dict.pkl', 'wb') as f: pickle.dump(Qe_dict, f)
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'wb') as f: pickle.dump(f_dict, f)


time: 0 ns (started: 2026-11-13 18:34:30 +01:00)


## Special transmon pin

In [ ]:
def pin_q_of_kappa(kappa, f):

    tmp = tul.transmon_search(7, f_dict['alpha'])
    _, _, n = tul.transmon(tmp[0], tmp[1])
    
    def loss(x):
        kappa_approx = tul.kappa_n(f, x)*np.abs(n[0, 1])**2
        return (kappa_approx - kappa)**2

    sol = scp.optimize.minimize_scalar(loss, bounds=[0, 100])
    return sol.x

pin_q_of_kappa = np.vectorize(pin_q_of_kappa)


### Modal Network

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Special_pin')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Special_pin')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)


# define pin length
hfss.variable_manager.set_variable(name="narrow_pin_h", expression=f"{-3.75}mm", overwrite=True)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['T_egm']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{5.65}GHz', RangeEnd=f'{5.8}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))'], primary_sweep_variable="Freq")
S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(freqs, np.abs(S), color='blue')

plt.xlabel('f, GHz')
plt.ylabel('S')
plt.title('S-parameter scan') 
plt.grid(True, alpha=0.3)
plt.legend()
plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# fit
pin = rfit.reflection_port()
pin.add_data(freqs*1e9, S)
pin.autofit()

pin.plotall()
display(pd.DataFrame([pin.fitresults]).applymap(lambda x: "{0:.2e}".format(x)))
print(f'bandwidth: {np.around(pin.fitresults['fr']/pin.fitresults['Qc']*1e-3, 4)} kHz\n')
print(f'T1: {np.around(1/(pin.fitresults['fr']/pin.fitresults['Qc']*2*np.pi)*1e3, 4)} ms\n')

f_dict.update({'T_via_pin_S':pin.fitresults['fr']})
Qe_dict.update({'T_via_pin_S':pin.fitresults['Qc']})
Qi_dict.update({'T_via_pin_S':pin.fitresults['Qi']})


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Special_pin')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Special_pin')

fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')

# add JJ inductance and capacitance
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=generic.constants.Plane.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

hfss.assign_lumped_rlc_to_sheet('JJ', start_direction=1, name='JJ_RLC', 
                                inductance=f'{tul.L_of_El(Ej_dict['T'])}n',
                                capacitance=f'{tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])}f')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['T_egm']}GHz')
setup.update_property('Passes', '5')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{5.6}GHz', RangeEnd=f'{5.9}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# sweep of height

h_list = np.linspace(-4.75, -3.75, 51)
Qi_list = np.zeros(h_list.shape[0])
Qe_list = np.zeros(h_list.shape[0])
Qe_err_list = np.zeros(h_list.shape[0])
f_list = np.zeros(h_list.shape[0])

for n in tqdm.tqdm(range(h_list.shape[0])):
    # define pin length
    hfss.variable_manager.set_variable(name="narrow_pin_h", expression=f"{h_list[n]}mm", overwrite=True)

    for m in range(3):
        # start simulation
        res = hfss.analyze(setup=hfss.get_setups()[0])
        if(res): break

    data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))'], primary_sweep_variable="Freq")
    S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
    freqs = np.asarray(data.primary_sweep_values, dtype=float) 

    # fit
    pin = rfit.reflection_port()
    pin.add_data(freqs*1e9, S)
    pin.autofit()
    
    Qi_list[n] = pin.fitresults['Qi']
    Qe_list[n] = pin.fitresults['Qc']
    f_list[n] = pin.fitresults['fr']
    Qe_err_list[n] = pin.fitresults['Qc_err']

    np.savez(data_path+'transmon_to_pin_coupling_S', h_list=h_list, f_list=f_list, Qi_list=Qi_list, Qe_list=Qe_list, Qe_err_list=Qe_err_list)
    

In [ ]:
data = np.load(data_path+'transmon_to_pin_coupling_S.npz')
h_list=data['h_list']
f_list=data['f_list']
Qi_list=data['Qi_list']
Qe_list=data['Qe_list']
Qe_err_list=data['Qe_err_list']

kappas = []
hs = []
Ts = []
Tis = []

# clean
for n in range(h_list.shape[0]):
    if(Qe_err_list[n]/Qe_list[n] < 0.01 and Qe_list[n]>100):
        kappas.append(f_list[n]/Qe_list[n])
        Ts.append(Qe_list[n]/f_list[n]/2/np.pi*1e3)
        hs.append(np.abs(h_list[n]))
        Tis.append(Qi_list[n]/f_list[n]/2/np.pi*1e3)

plt.figure(figsize=(8, 6))
plt.plot(hs, kappas, label='kappas', color='black', linewidth=3)
plt.xlabel('h, mm')
plt.ylabel('kappa, Hz')
plt.title(f'Kappa Hz') 
plt.grid(True, alpha=0.3)
plt.legend()


plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
# plt.plot(hs, kappas, label='kappas', color='black', linewidth=3)
plt.plot(hs, Ts, label='Te', color='green', linewidth=3)
plt.plot(hs, Tis, label='Ti', color='blue', linewidth=3)

plt.xlabel('h, mm')
plt.ylabel('T, ms')
plt.title(f'T limits,ms') 
plt.grid(True, alpha=0.3)
plt.legend()
# plt.xlim([0, 1e-4])
# plt.ylim([1e-1, 1.2])
# plt.xlim([8, 8.5])
plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


### Q3D

In [ ]:
q3d = Q3d(version=ansys_version, project='Sisyphus', design='Capacitance_special_pin')

In [ ]:
fourq_q3d = design.renderers.q3d
fourq_q3d.start()
fourq_q3d.activate_ansys_design('Capacitance_special_pin')


In [ ]:
# add transmon to design
fourq_q3d.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)

# delete autoground
q3d.modeler.delete('ground_main_plane')

In [ ]:
# define pin length
q3d.variable_manager.set_variable(name="narrow_pin_h", expression=f"{-3.4}mm", overwrite=True)

In [ ]:
q3d.analyze('Setup')

In [ ]:
C_lom = fourq_q3d.get_capacitance_matrix()
display(C_lom[0])
print('unit:', C_lom[1])


In [ ]:
# order island names
island_sequence = []
island_sequence.append('pad_bot_T -> T_bot')
island_sequence.append('pad_top_T -> T_top')

# order pins names
pins = []
pins.append('Pin -> pin')

# conver C_lom to fancy form
C_df, C_pins_df = tul.lom_c_matrix_convert(C_lom[0], island_sequence, pins=pins, physical=False)

display(C_df)
display(C_pins_df)

C = np.asarray(C_df)
C_pins = np.asarray(C_pins_df)

# add jj capacitance
C[0, 1] += tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])


In [ ]:
# transfer matrix
S = np.eye(C.shape[0])

# first mode will be amtisym = transmon
S[0, 1] = -1
S[1, 0] = 1

Ec, pin_q = tul.legendre_forward(C, C_pins=C_pins, S=S)
print(f'Transmon Ec = {np.around(Ec[0, 0], 3)} GHz')
s, _, n = tul.transmon(Ej_dict['T'], Ec[0, 0], truncation=20)
print(f'f01: {np.around(s[1], 3)} GHz\nalpha: {np.around(s[2] - 2*s[1], 3)} GHz')
print(f'pin_q: {np.around(pin_q[0, 0], 3)} GHz/mV')
print(f'T1: {np.around(1/(2*np.pi*tul.kappa_n(s[1], pin_q[0, 0])*np.abs(n[0,1])**2), 3)} ns')


In [ ]:
# delete old qiskit design
q3d.modeler.delete(q3d.modeler.get_objects_in_group('Sheets'))
q3d.modeler.delete('main')

# break connection with ANSYS
fourq_q3d.stop()
q3d.release_desktop(close_projects=False, close_desktop=False)


In [ ]:
q3d = Q3d(version=ansys_version, project='Sisyphus', design='Capacitance_special_pin')

fourq_q3d = design.renderers.q3d
fourq_q3d.start()
fourq_q3d.activate_ansys_design('Capacitance_special_pin')

# add transmon to design
fourq_q3d.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)

# delete autoground
q3d.modeler.delete('ground_main_plane')

h_list = np.linspace(-4.75, -2.75, 51)
Ec_list = np.zeros(h_list.shape[0])
pin_q_list = np.zeros(h_list.shape[0])

for i in tqdm.tqdm(range(h_list.shape[0])):
    # define pin length
    q3d.variable_manager.set_variable(name="narrow_pin_h", expression=f"{h_list[i]}mm", overwrite=True)
    for k in range(3):
        res=q3d.analyze('Setup')
        if(res): break

    C_lom = fourq_q3d.get_capacitance_matrix()
    # order island names
    island_sequence = []
    island_sequence.append('pad_bot_T -> T_bot')
    island_sequence.append('pad_top_T -> T_top')
    
    # order pins names
    pins = []
    pins.append('Pin -> pin')
    
    # conver C_lom to fancy form
    C_df, C_pins_df = tul.lom_c_matrix_convert(C_lom[0], island_sequence, pins=pins, physical=False)
    
    C = np.asarray(C_df)
    C_pins = np.asarray(C_pins_df)
    
    # add jj capacitance
    C[0, 1] += tul.C_of_Ej(Ej_dict['T'], Ej_to_S=exp_vars['Ej_to_S'])
    
    # transfer matrix
    S = np.eye(C.shape[0])
    
    # first mode will be amtisym = transmon
    S[0, 1] = -1
    S[1, 0] = 1
    
    Ec, pin_q = tul.legendre_forward(C, C_pins=C_pins, S=S)
    Ec_list[i] = Ec[0, 0]
    pin_q_list[i] = pin_q[0, 0]
    
    np.savez(data_path+'transmon_to_pin_coupling_Q3D', h_list=h_list, Ec_list=Ec_list, pin_q_list=pin_q_list)
    

In [ ]:
data = np.load(data_path+'transmon_to_pin_coupling_Q3D.npz')
h_list=data['h_list']
Ec_list=data['Ec_list']
pin_q_list=data['pin_q_list']

kappas = np.zeros(h_list.shape[0])
Ts = np.zeros(h_list.shape[0])
# clean
for i in range(h_list.shape[0]):
    s, _, n = tul.transmon(Ej_dict['T'], Ec_list[i])
    kappas[i] = tul.kappa_n(s[1], pin_q_list[i])*np.abs(n[0, 1])**2*1e9
    Ts[i] = 1/kappas[i]/2/np.pi*1e3

plt.figure(figsize=(8, 6))
plt.plot(np.abs(h_list), kappas, label='kappas', color='black', linewidth=3)
plt.xlabel('h, mm')
plt.ylabel('kappa, Hz')
plt.title(f'Kappa Hz') 
plt.grid(True, alpha=0.3)
plt.legend()
plt.yscale('log')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
# plt.plot(hs, kappas, label='kappas', color='black', linewidth=3)
plt.plot(np.abs(h_list), Ts, label='Te', color='green', linewidth=3)

plt.xlabel('h, mm')
plt.ylabel('T, ms')
plt.title(f'T limits,ms') 
plt.grid(True, alpha=0.3)
plt.legend()
# plt.xlim([0, 1e-4])
# plt.ylim([1e-1, 1.2])
# plt.xlim([8, 8.5])
plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# delete old qiskit design
q3d.modeler.delete(q3d.modeler.get_objects_in_group('Sheets'))
q3d.modeler.delete('main')

# break connection with ANSYS
fourq_q3d.stop()
q3d.release_desktop(close_projects=False, close_desktop=False)


### Y-port

In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Special_pin')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Special_pin')


In [ ]:
fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=1, name='JJ_port')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)


# define pin length
hfss.variable_manager.set_variable(name="narrow_pin_h", expression=f"{-3.4}mm", overwrite=True)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['T_egm']}GHz')
setup.update_property('Passes', '15')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{f_dict['T_egm']-0.3}GHz', RangeEnd=f'{f_dict['T_egm']+0.3}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# start simulation
hfss.analyze(setup=hfss.get_setups()[0])

In [ ]:
data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))',
                                                're(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
freqs = np.asarray(data.primary_sweep_values, dtype=float) 


In [ ]:
tmp = tul.bbq_resonances_of_Y(freqs, Y, tul.L_of_El(Ej_dict['T']), C=tul.C_of_Ej(Ej_dict['T']), kappas=True)
f = tmp[0][0]
kappa = tmp[1][0]
pin_q = pin_q_of_kappa(kappa, f)
print(f'frequency: {np.around(f, 3)} GHz\npin_q: {np.around(pin_q, 3)} GHz/mV\nT1 limit: {np.around(1/kappa/2/np.pi*1e-6, 3)} ms')

Ec_dict.update({'pin_q_narrow':pin_q})
f_dict.update({'T_pin_Y':f})


In [ ]:
Ls = []
kappas = []
frs = []

for L in np.linspace(-0.2, 0.2, 101):
    tmp = tul.bbq_resonances_of_Y(freqs, Y, tul.L_of_El(Ej_dict['T'])+L, C=tul.C_of_Ej(Ej_dict['T']), kappas=True)
    try:
        frs.append(tmp[0])
        kappas.append(tmp[1])
        Ls.append(L)
    except:
        pass
    

In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(frs, np.asarray(kappas)*1e6, color='blue')

plt.xlabel('frequency, GHz')
plt.ylabel('kappa, kHz')
plt.title('Kappa scan') 
plt.grid(True, alpha=0.3)
plt.legend()
# plt.yscale('log')

plt.tight_layout()
plt.show()


In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


In [ ]:
# connect to HFSS
hfss = Hfss(version=ansys_version, project='Sisyphus', design='Special_pin')

fourq_hfss = design.renderers.hfss
fourq_hfss.start()
fourq_hfss.activate_ansys_design('Special_pin')

fourq_hfss.render_design(['T', 'taper_p1', 'taper_p2'], box_plus_buffer=False)
hfss.modeler.delete('ground_main_plane')
hfss.modeler.delete('sample_holder')


# add JJ
gap_mm = taper_gap/1e3+ 2*float(length_cut_taper[:-2])/1e3
rect = hfss.modeler.create_rectangle(orientation=hfss.PLANE.XY,
                                     origin=[0 - 0.022/2, float(T_y[:-2])/1e3 - gap_mm/2, 0],
                                     sizes = ['0.022mm', f'{gap_mm}mm'],
                                     name='JJ',
                                     is_covered=True,
                                    )

# assign lumped port to JJ sheet
hfss.lumped_port('JJ', integration_line=1, name='JJ_port')

#  assign the JJ mesh
hfss.mesh.assign_length_mesh(['JJ', 'taper_main_taper_p1', 'taper_main_taper_p2'], inside_selection=False, maximum_length="30um", 
                             name='transmon_mesh', maximum_elements=None,)

# create sweep
setup = hfss.get_setup(hfss.get_setups()[0])
setup.update_property('Solution Freq', f'{f_dict['T_egm']}GHz')
setup.update_property('Passes', '15')
hfss.oanalysis.DeleteSweep(setup.name, setup.get_sweep_names()[0])
sweep = setup.add_sweep(name='Sweep', sweep_type='Interpolating')
sweep.props.update(RangeStart=f'{f_dict['T_egm']-0.3}GHz', RangeEnd=f'{f_dict['T_egm']+0.3}GHz', RangeCount=24001, InterpTolerance=0.01, InterpMaxSolns=3000)
sweep.update()


In [ ]:
# sweep of height

h_list = np.linspace(-4.75, -3.25, 101)
kappa_list = np.zeros(h_list.shape[0])
f_list = np.zeros(h_list.shape[0])

for n in tqdm.tqdm(range(h_list.shape[0])):
    # define pin length
    hfss.variable_manager.set_variable(name="narrow_pin_h", expression=f"{h_list[n]}mm", overwrite=True)

    for m in range(3):
        # start simulation
        res = hfss.analyze(setup=hfss.get_setups()[0])
        if(res): break

    data = hfss.post.get_solution_data(expressions=['re(S(line_port,line_port))', 'im(S(line_port,line_port))',
                                                    're(Y(JJ_port,JJ_port))', 'im(Y(JJ_port,JJ_port))'], primary_sweep_variable="Freq")
    Y = data.data_real("re(Y(JJ_port,JJ_port))")+1j*data.data_real("im(Y(JJ_port,JJ_port))")
    S = data.data_real("re(S(line_port,line_port))")+1j*data.data_real("im(S(line_port,line_port))")
    freqs = np.asarray(data.primary_sweep_values, dtype=float)

    # fit
    tmp = tul.bbq_resonances_of_Y(freqs, Y, tul.L_of_El(Ej_dict['T'])+L, C=tul.C_of_Ej(Ej_dict['T']), kappas=True)
    try:
        f_list[n] = tmp[0][0]
        kappa_list[n] = tmp[1][0]
    except:
        pass

    np.savez(data_path+'transmon_to_pin_coupling_Y', h_list=h_list, f_list=f_list, kappa_list=kappa_list)
    

In [ ]:
# delete old qiskit design
hfss.modeler.delete(hfss.modeler.get_objects_in_group('Perfect E'))
hfss.modeler.delete('main')
hfss.modeler.delete('JJ')

# break connection with ANSYS
fourq_hfss.stop()
hfss.release_desktop(close_projects=False, close_desktop=False)


In [ ]:
data = np.load(data_path+'transmon_to_pin_coupling_Y.npz')
h_list=data['h_list']
f_list=data['f_list']
kappa_list=data['kappa_list']

plt.figure(figsize=(8, 6))
plt.plot(np.abs(h_list), kappa_list*1e9, label='kappas', color='black', linewidth=3)
plt.xlabel('h, mm')
plt.ylabel('kappa, Hz')
plt.title(f'Kappa Hz') 
plt.grid(True, alpha=0.3)
plt.legend()
plt.yscale('log')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
# plt.plot(hs, kappas, label='kappas', color='black', linewidth=3)
plt.plot(np.abs(h_list), 1/kappa_list/2/np.pi*1e-6, label='Te', color='green', linewidth=3)

plt.xlabel('h, mm')
plt.ylabel('T, ms')
plt.title(f'T limits,ms') 
plt.grid(True, alpha=0.3)
plt.legend()
# plt.xlim([0, 1e-4])
# plt.ylim([1e-1, 1.2])
# plt.xlim([8, 8.5])
# plt.yscale('log')

plt.tight_layout()
plt.show()


### Merged plotting

In [ ]:
data = np.load(data_path+'transmon_to_pin_coupling_Q3D.npz')
h_list=data['h_list']
Ec_list=data['Ec_list']
pin_q_list=data['pin_q_list']

pin_q = np.zeros(h_list.shape[0])
kappas = np.zeros(h_list.shape[0])
Ts = np.zeros(h_list.shape[0])
# clean
for i in range(h_list.shape[0]):
    s, _, n = tul.transmon(Ej_dict['T'], Ec_list[i])
    kappas[i] = tul.kappa_n(s[1], pin_q_list[i])*np.abs(n[0, 1])**2*1e9
    Ts[i] = 1/kappas[i]/2/np.pi*1e3
    pin_q[i] = pin_q_of_kappa(kappas[i]/1e9, s[1])


data = np.load(data_path+'transmon_to_pin_coupling_Y.npz')
h_list_Y=data['h_list']
f_list_Y=data['f_list']
kappa_list_Y=data['kappa_list']
pin_q_Y = pin_q_of_kappa(kappa_list_Y, f_list_Y)


In [ ]:
plt.figure(figsize=(8, 6))
plt.plot(np.abs(5.15 + h_list), kappas, label='DC capacitance', color='black', linewidth=2)
plt.plot(np.abs(5.15 + h_list_Y), kappa_list_Y*1e9, label='Y-simulation', color='green', linewidth=2)
plt.xlabel('distance from wafer, mm')
plt.ylabel('kappa, Hz')
plt.title(f'Kappa, Hz') 
plt.grid(True, alpha=0.3)
plt.legend()
plt.yscale('log')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(np.abs(5.15 + h_list), Ts, label='DC capacitance', color='black', linewidth=2)
plt.plot(np.abs(5.15 + h_list_Y), 1/kappa_list_Y/2/np.pi*1e-6, label='Y-simulation', color='green', linewidth=2)
plt.xlabel('distance from wafer, mm')
plt.ylabel('T, ms')
plt.title(f'T limits, ms') 
plt.grid(True, alpha=0.3)
plt.legend()
# plt.xlim([0, 1e-4])
# plt.ylim([1e-1, 1.2])
# plt.xlim([8, 8.5])
plt.yscale('log')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 6))
plt.plot(np.abs(5.15 + h_list), pin_q, label='DC capacitance', color='black', linewidth=2)
plt.plot(np.abs(5.15 + h_list_Y), pin_q_Y, label='Y-simulation', color='green', linewidth=2)
plt.xlabel('distance from wafer, mm')
plt.ylabel('pin_q, GHz/mV')
plt.title(f'pin_q estimation GHz/mV') 
plt.grid(True, alpha=0.3)
plt.legend()
# plt.xlim([0, 1e-4])
# plt.ylim([1e-1, 1.2])
# plt.xlim([8, 8.5])
plt.yscale('log')
plt.tight_layout()
plt.show()


### Save

In [ ]:
with open(data_path + 'Sisyphus_A_Ec_dict.pkl', 'wb') as f: pickle.dump(Ec_dict, f)
with open(data_path + 'Sisyphus_A_f_dict.pkl', 'wb') as f: pickle.dump(f_dict, f)


# Summary

In [20]:
# init
print(f'Transmon Ec = {np.around(Ec_dict['Ec_T'][0, 0], 3)} GHz')
s, _, n = tul.transmon(Ej_dict['T'], Ec_dict['Ec_T'][0, 0], truncation=20)
print(f'f01: {np.around(s[1], 3)} GHz\nalpha: {np.around(s[2] - 2*s[1], 3)} GHz')
print(f'pin_q: {np.around(Ec_dict['pin_q'][0, 0], 3)} GHz/mV')
print(f'fat pin decay: {np.around(1/(2*np.pi*tul.kappa_n(s[1], Ec_dict['pin_q'][0, 0])*np.abs(n[0, 1])**2)/1e6, 3)} ms')


Transmon Ec = 0.207 GHz
f01: 6.096 GHz
alpha: -0.226 GHz
pin_q: 0.11 GHz/mV
fat pin decay: 2.826 ms
time: 0 ns (started: 2026-11-13 17:50:04 +01:00)


## Chi

### Filter

In [21]:
g = g_dict['T-filt']
freq_f = f_dict['filter_bare']
kappa_f = freq_f/Qe_dict['filter_via_pin_S']
s0, a, at = tul.oscillator(freq_f)
sm, (key, selfhood) = tul.mix_two_sys(s, s0, n, g*(a+at), truncation=20)
print(f'chi: {(sm[key[1, 1]] - sm[key[0, 1]] - sm[key[1, 0]])/2*1e6} kHz')
kappa = tul.kappa_osc_via_osc_eq(s0[1], s[1], g, kappa_f, regime='cut')
print(f'purcell limit: {1/2/np.pi/kappa*1e-6} ms')


chi: -0.10814979045647988 kHz
purcell limit: 42.05338582569959 ms
time: 140 ms (started: 2026-11-13 17:50:04 +01:00)


### Read

In [22]:
# qubit
g = g_dict['T-read']
freq_r = f_dict['filter_bare']
kappa_r = freq_r/Qe_dict['read_via_pin_S']
s0, a, at = tul.oscillator(freq_r)
sm, (key, selfhood) = tul.mix_two_sys(s, s0, n, g*(a+at), truncation=20)
print(f'chi: {(sm[key[1, 1]] - sm[key[0, 1]] - sm[key[1, 0]])/2*1e3} MHz')
kappa = tul.kappa_osc_via_osc_eq(s0[1], s[1], g, kappa_r, regime='cut')
print(f'direct purcell limit: {1/2/np.pi/kappa*1e-6} ms')


chi: -1.503797611008828 MHz
direct purcell limit: 56.203387666723735 ms
time: 125 ms (started: 2026-11-13 17:50:04 +01:00)


In [23]:
# filter
g = g_dict['read-filter']
s0, a, at = tul.oscillator(freq_r)
s1, b, bt = tul.oscillator(freq_f)
kappa = tul.kappa_osc_via_osc_eq(freq_r, freq_f, g, kappa_f, regime='cut')
print(f'direct purcell limit: {kappa*1e3} MHz')


direct purcell limit: 5.251659005310622 MHz
time: 0 ns (started: 2026-11-13 17:50:05 +01:00)


### Storage

In [24]:
# qubit
g = g_dict['T-stor']
freq_s = f_dict['storage']
s0, a, at = tul.oscillator(freq_s)
sm, (key, selfhood) = tul.mix_two_sys(s, s0, n, g*(a+at), truncation=20)
print(f'chi: {(sm[key[1, 1]] - sm[key[0, 1]] - sm[key[1, 0]])/2*1e6} kHz')


chi: -44.21338380611317 kHz
time: 110 ms (started: 2026-11-13 17:50:05 +01:00)


In [25]:
# fat C-sim
pin_q = Ec_dict['pin_q'][0, 0]
kappa_q = tul.kappa_n(s[1], pin_q)*np.abs(n[0, 1])**2
kappa = tul.kappa_osc_via_osc_eq(s[1], freq_s, g, kappa_q, regime='cut')
print(f'direct purcell limit: {1/2/np.pi/kappa*1e-6} ms')


direct purcell limit: 77693.88477600201 ms
time: 0 ns (started: 2026-11-13 17:50:05 +01:00)


In [26]:
# thin C-sim
pin_q = 1.258
kappa_q = tul.kappa_n(s[1], pin_q)*np.abs(n[0, 1])**2
kappa = tul.kappa_osc_via_osc_eq(s[1], freq_s, g, kappa_q, regime='cut')
print(f'direct purcell limit: {1/2/np.pi/kappa*1e-6} ms')


direct purcell limit: 593.2261133088001 ms
time: 16 ms (started: 2026-11-13 17:50:05 +01:00)


In [27]:
# thin Y-sim
pin_q = 0.087
kappa_q = tul.kappa_n(s[1], pin_q)*np.abs(n[0, 1])**2
kappa = tul.kappa_osc_via_osc_eq(s0[1], s[1], g, kappa_q, regime='cut')
print(f'direct purcell limit: {1/2/np.pi/kappa*1e-6} ms')


direct purcell limit: 10495.695948151917 ms
time: 0 ns (started: 2026-11-13 17:50:05 +01:00)
